# Gold Layer Implementation

Builds business-ready aggregate tables from **silver** (never from bronze). Every table is a full refresh
(`CREATE OR REPLACE TABLE ... AS SELECT`): gold is small and derived, so rebuilding it is simpler and always
consistent with silver, and re-running the notebook is safe.

| Gold table | Business question |
|---|---|
| `gold.sales_by_policy_type_and_month` | How much premium do we sell per policy type each month? |
| `gold.claims_by_policy_type_and_status` | How many claims, and how much money, per policy type and claim status? |
| `gold.claim_analysis_by_policy_type` | COUNT / AVG / MIN / MAX of claims per policy type, plus which types are profitable (loss ratio) |

**Job order:** runs last, after Silver_Policy and Silver_Claim.

**Known gap:** the source has no link from a policy to an agent or branch, so branch / agent performance
cannot be reported until the policy feed carries `agent_id`.

In [0]:
%sql
CREATE OR REPLACE TABLE policysystemdatabricks.gold.sales_by_policy_type_and_month
COMMENT 'Premium sold per policy type per month (policy start month). Source: silver.policydata.'
AS
SELECT
  policy_type,
  CAST(date_trunc('MONTH', start_date) AS DATE)          AS sales_month,
  count(*)                                              AS policies_sold,
  CAST(sum(premium) AS DECIMAL(18,2))                   AS total_premium,
  CAST(avg(premium) AS DECIMAL(18,2))                   AS avg_premium,
  CAST(sum(coverage_amount) AS DECIMAL(18,2))           AS total_coverage_amount,
  current_timestamp()                                   AS refreshed_at
FROM policysystemdatabricks.silver.policydata
GROUP BY policy_type, CAST(date_trunc('MONTH', start_date) AS DATE)

In [0]:
%sql
CREATE OR REPLACE TABLE policysystemdatabricks.gold.claims_by_policy_type_and_status
COMMENT 'Claim count and amount per policy type and claim status. Source: silver.claim + silver.policydata.'
AS
SELECT
  p.policy_type,
  c.claim_status,
  count(*)                                              AS claim_count,
  CAST(sum(c.claim_amount) AS DECIMAL(18,2))            AS total_claim_amount,
  CAST(avg(c.claim_amount) AS DECIMAL(18,2))            AS avg_claim_amount,
  min(c.date_of_claim)                                  AS first_claim_date,
  max(c.date_of_claim)                                  AS last_claim_date,
  current_timestamp()                                   AS refreshed_at
FROM policysystemdatabricks.silver.claim c
JOIN policysystemdatabricks.silver.policydata p ON c.policy_id = p.policy_id
GROUP BY p.policy_type, c.claim_status

In [0]:
%sql
-- Claim statistics and premium are aggregated SEPARATELY, then joined on policy_type.
-- Joining raw claims to policies and summing premium would repeat a policy's premium once per claim (fan-out).
CREATE OR REPLACE TABLE policysystemdatabricks.gold.claim_analysis_by_policy_type
COMMENT 'Claim count / avg / min / max per policy type, plus claim frequency and loss ratio (claims paid / premium earned).'
AS
WITH claim_stats AS (           -- one row per policy type, computed over individual claims
  SELECT p.policy_type,
         count(*)                    AS claim_count,
         count(DISTINCT c.policy_id) AS policies_with_claims,
         sum(c.claim_amount)         AS total_claim_amount,
         avg(c.claim_amount)         AS avg_claim_amount,
         min(c.claim_amount)         AS min_claim_amount,
         max(c.claim_amount)         AS max_claim_amount
  FROM policysystemdatabricks.silver.claim c
  JOIN policysystemdatabricks.silver.policydata p ON c.policy_id = p.policy_id
  GROUP BY p.policy_type
),
policy_stats AS (               -- one row per policy type, computed over policies
  SELECT policy_type, count(*) AS policies, sum(premium) AS total_premium
  FROM policysystemdatabricks.silver.policydata
  GROUP BY policy_type
)
SELECT
  ps.policy_type,
  coalesce(cs.claim_count, 0)                                       AS claim_count,
  CAST(cs.avg_claim_amount AS DECIMAL(18,2))                        AS avg_claim_amount,
  CAST(cs.min_claim_amount AS DECIMAL(18,2))                        AS min_claim_amount,
  CAST(cs.max_claim_amount AS DECIMAL(18,2))                        AS max_claim_amount,
  CAST(coalesce(cs.total_claim_amount, 0) AS DECIMAL(18,2))         AS total_claim_amount,
  ps.policies,
  coalesce(cs.policies_with_claims, 0)                              AS policies_with_claims,
  CAST(ps.total_premium AS DECIMAL(18,2))                           AS total_premium,
  round(coalesce(cs.policies_with_claims, 0) / ps.policies, 4)      AS claim_frequency,  -- share of policies with >= 1 claim
  round(coalesce(cs.total_claim_amount, 0) / ps.total_premium, 4)   AS loss_ratio,       -- > 1 = paying out more than earned
  current_timestamp()                                               AS refreshed_at
FROM policy_stats ps
LEFT JOIN claim_stats cs ON ps.policy_type = cs.policy_type

In [0]:
%sql
-- Gold must add up to silver. Any non-zero value means a join or grouping is dropping or duplicating rows.
SELECT
  (SELECT count(*) FROM policysystemdatabricks.silver.policydata)
    - (SELECT sum(policies_sold) FROM policysystemdatabricks.gold.sales_by_policy_type_and_month)      AS sales_policy_diff,
  (SELECT CAST(sum(premium) AS DECIMAL(18,2)) FROM policysystemdatabricks.silver.policydata)
    - (SELECT sum(total_premium) FROM policysystemdatabricks.gold.claim_analysis_by_policy_type)       AS premium_diff,
  (SELECT count(*) FROM policysystemdatabricks.silver.claim)
    - (SELECT sum(claim_count) FROM policysystemdatabricks.gold.claims_by_policy_type_and_status)     AS status_claim_diff,
  (SELECT count(*) FROM policysystemdatabricks.silver.claim)
    - (SELECT sum(claim_count) FROM policysystemdatabricks.gold.claim_analysis_by_policy_type)        AS analysis_claim_diff

In [0]:
%sql
SELECT policy_type, claim_count, avg_claim_amount, min_claim_amount, max_claim_amount,
       total_claim_amount, total_premium, claim_frequency, loss_ratio
FROM policysystemdatabricks.gold.claim_analysis_by_policy_type
ORDER BY loss_ratio DESC